# Cell2Location: Xenium Mouse Brain
**Dataset:** Xenium V1 FF Mouse Brain Coronal Subset (CTX + HP)

**Workflow:**
1. Load Xenium spatial data
2. Download Allen Brain scRNA-seq reference atlas
3. Train regression model to get cell type signatures
4. Run spatial deconvolution model
5. Visualize cell type abundances spatially

**Environment:** Run in the SCC Desktop Singularity session (`cellpymc` conda env)

> **Note:** This notebook is configured for fast testing (`n_iter=100` for regression, `n_iter=500` for spatial, `n_comb=20`). For final results, increase to `n_iter=4000` and `n_iter=20000`. Use a GPU in the singularity environment.

In [1]:
# Must be set BEFORE any theano/pymc3 imports
import os
os.environ["THEANO_FLAGS"] = "device=cuda,floatX=float32"

## 0. Imports

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import urllib.request
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Paths
DATA_DIR = "/projectnb/ds596/projects/Team_7/data/xeniumMouseBrain"
RESULTS_DIR = "/projectnb/ds596/projects/Team_7/cploss_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

ref_h5ad_path = os.path.join(RESULTS_DIR, "all_cells_20200625.h5ad")
ref_csv_path = os.path.join(RESULTS_DIR, "snRNA_annotation_astro_subtypes_refined59_20200823.csv")

print("Imports done.")
print(f"Data dir:    {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")

Imports done.
Data dir:    /projectnb/ds596/projects/Team_7/data/xeniumMouseBrain
Results dir: /projectnb/ds596/projects/Team_7/cploss_results


## 1. Load Xenium Spatial Data

In [3]:
# Load cell-by-gene count matrix
adata_xenium = sc.read_10x_h5(
    os.path.join(DATA_DIR, "cell_feature_matrix.h5")
)
adata_xenium.var_names_make_unique()

# Add spatial coordinates
cells = pd.read_csv(os.path.join(DATA_DIR, "cells.csv.gz"))
cells = cells.set_index("cell_id")
cells.index = cells.index.astype(str)  # match adata string index

adata_xenium.obs = adata_xenium.obs.join(cells[["x_centroid", "y_centroid"]], how="left")
adata_xenium.obsm["spatial"] = adata_xenium.obs[["x_centroid", "y_centroid"]].values

print(adata_xenium)
print(f"\nSpatial coords loaded: {adata_xenium.obs[['x_centroid','y_centroid']].head()}")

AnnData object with n_obs × n_vars = 36602 × 248
    obs: 'x_centroid', 'y_centroid'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'

Spatial coords loaded:     x_centroid   y_centroid
1  1898.814917  2540.963147
2  1895.304724  2532.626880
3  2368.073230  2534.409265
4  1903.726282  2560.009912
5  1917.481299  2543.131934


## 2. Download cell2location scRNA-seq Reference Atlas
This is the official reference used in the cell2location mouse brain tutorial (~700MB, may take a few minutes).

In [4]:
ref_h5ad_path = os.path.join(RESULTS_DIR, "all_cells_20200625.h5ad")
ref_csv_path = os.path.join(RESULTS_DIR, "snRNA_annotation_astro_subtypes_refined59_20200823.csv")

# Download reference h5ad (~700MB) — skip if already downloaded
if not os.path.exists(ref_h5ad_path):
    print("Downloading reference h5ad (~700MB)...")
    urllib.request.urlretrieve(
        "https://cell2location.cog.sanger.ac.uk/tutorial/mouse_brain_snrna/all_cells_20200625.h5ad",
        ref_h5ad_path
    )
    print("Done.")
else:
    print("Reference h5ad already exists, skipping download.")

# Download annotation CSV — skip if already downloaded
if not os.path.exists(ref_csv_path):
    print("Downloading annotation CSV...")
    urllib.request.urlretrieve(
        "https://cell2location.cog.sanger.ac.uk/tutorial/mouse_brain_snrna/snRNA_annotation_astro_subtypes_refined59_20200823.csv",
        ref_csv_path
    )
    print("Done.")
else:
    print("Annotation CSV already exists, skipping download.")

Reference h5ad already exists, skipping download.
Annotation CSV already exists, skipping download.


In [5]:
# Load reference and merge cell type annotations
adata_ref = sc.read_h5ad(ref_h5ad_path)

ann = pd.read_csv(ref_csv_path, index_col=0)
adata_ref.obs = adata_ref.obs.join(ann, how="left")

print(adata_ref)
print(f"\nCell type counts:")
print(adata_ref.obs["annotation_1"].value_counts().head(10))

AnnData object with n_obs × n_vars = 40572 × 31053
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'sample', 'barcode', 'annotation_1'
    var: 'feature_types', 'genome', 'SYMBOL', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

Cell type counts:
Oligo_2        10819
Ext_L56         1422
Inh_4           1389
Ext_L23         1244
Micro           1218
Ext_Thal_1      1197
OPC_1           1015
Ext_Pir          954
Ext_L25          922
Ext_Hpc_DG2      915
Name: annotation_1, dtype: int64


## 3. Find Shared Genes & QC

In [6]:
# Fix reference gene names: swap Ensembl IDs -> gene symbols
adata_ref.var["SYMBOL"] = adata_ref.var["SYMBOL"].astype(str)
adata_ref.var_names = adata_ref.var["SYMBOL"]
adata_ref.var_names_make_unique()

print("Reference genes (first 5):", adata_ref.var_names[:5].tolist())
print("Xenium genes (first 5):", adata_xenium.var_names[:5].tolist())

# Find shared genes
shared_genes = adata_xenium.var_names.intersection(adata_ref.var_names)
print(f"\nShared genes: {len(shared_genes)}")

# Subset both to shared genes
adata_xenium = adata_xenium[:, shared_genes].copy()
adata_ref = adata_ref[:, shared_genes].copy()

print(f"Xenium shape: {adata_xenium.shape}")
print(f"Reference shape: {adata_ref.shape}")

Reference genes (first 5): ['Xkr4', 'Gm1992', 'Gm37381', 'Rp1', 'Sox17']
Xenium genes (first 5): ['2010300C02Rik', 'Acsbg1', 'Acta2', 'Acvrl1', 'Adamts2']

Shared genes: 247
Xenium shape: (36602, 247)
Reference shape: (40572, 247)


In [7]:
# Basic QC on Xenium data
sc.pp.filter_cells(adata_xenium, min_counts=10)
sc.pp.filter_genes(adata_xenium, min_cells=5)

print(f"Xenium after QC: {adata_xenium.shape}")

Xenium after QC: (36419, 247)


## 4. Train Regression Model on Reference (Cell Type Signatures)
This uses PyTorch and runs fast. For testing: `n_iter=100`. For final results: `n_iter=4000`.

In [ ]:
from cell2location.models import RegressionGeneBackgroundCoverageTorch

# Prepare inputs
cell2covar = pd.DataFrame({
    "sample": adata_ref.obs["sample"].values,
    "annotation_1": adata_ref.obs["annotation_1"].values
}, index=adata_ref.obs_names)

X_data_ref = adata_ref.X.toarray() if hasattr(adata_ref.X, "toarray") else np.array(adata_ref.X)

# ---- SPEED SETTINGS ----
# Testing:  n_iter=100
# Final:    n_iter=4000
N_ITER_REG = 4000
# ------------------------

mod = RegressionGeneBackgroundCoverageTorch(
    sample_id="sample",
    cell2covar=cell2covar,
    X_data=X_data_ref,
    n_iter=N_ITER_REG,
    learning_rate=0.005,
    use_cuda=False,
    var_names=adata_ref.var_names.tolist(),
    obs_names=adata_ref.obs_names.tolist()
)

print("Model built successfully!")
print(f"n_obs: {mod.n_obs}, n_var: {mod.n_var}, n_fact: {mod.n_fact}")

Using cuDNN version 7605 on context None
Mapped name None to device cuda: Tesla V100-SXM2-16GB (0000:AF:00.0)


Model built successfully!
n_obs: 40572, n_var: 247, n_fact: 65


In [ ]:
# Train regression model
mod.fit_advi_iterative(n=1, n_iter=N_ITER_REG, train_proportion=0.9)
print("Regression model trained!")

In [ ]:
# Extract cell type expression signatures
mod.sample_posterior(node='all', n_samples=1000, save_samples=False, mean_field_slot='init_1')

inf_aver = mod.samples['post_sample_means']['gene_factors']
inf_aver = pd.DataFrame(inf_aver,
                         index=mod.fact_names,
                         columns=mod.var_names).T

# Keep only cell type columns (drop sample/batch columns)
inf_aver = inf_aver.loc[:, ~mod.which_sample.values]

# Clean up column names
inf_aver.columns = [c.replace("annotation_1_", "") for c in inf_aver.columns]

print(f"Cell type signatures shape: {inf_aver.shape}")
print(f"Cell types: {inf_aver.columns.tolist()[:10]}...")

## 5. Run Spatial Deconvolution Model on Xenium Data
This uses PyMC3/Theano and is slower. For testing: `n_iter=500`, `n_comb=20`. For final: `n_iter=20000`, `n_comb=50`.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU available on this node")

from cell2location.models import LocationModelLinearDependentWMultiExperiment

X_data_xenium = adata_xenium.X.toarray() if hasattr(adata_xenium.X, "toarray") else np.array(adata_xenium.X)

# ---- SPEED SETTINGS ----
# Testing:  n_iter=500,   n_comb=20
# Final:    n_iter=20000, n_comb=50
N_ITER_SPATIAL = 20000
N_COMB = 50
# ------------------------

mod_spatial = LocationModelLinearDependentWMultiExperiment(
    cell_state_mat=inf_aver.values,
    X_data=X_data_xenium,
    n_comb=N_COMB,
    n_iter=N_ITER_SPATIAL,
    learning_rate=0.005,
    sample_id=np.array(["xenium_sample"] * adata_xenium.n_obs),
    var_names=adata_xenium.var_names.tolist(),
    obs_names=adata_xenium.obs_names.tolist(),
    fact_names=inf_aver.columns.tolist()
)

print("Spatial model built!")
print(f"n_obs: {mod_spatial.n_obs}, n_var: {mod_spatial.n_var}, n_fact: {mod_spatial.n_fact}")

In [ ]:
# Train spatial model (slow — ~30 min at 500 iter on CPU)
mod_spatial.fit_advi(n=1, n_type='restart')
print("Spatial model trained!")

In [ ]:
# Sample posterior — use small n_samples + large batch_size for speed
mod_spatial.sample_posterior(
    node='all',
    n_samples=200,       # reduced from 1000 for speed
    save_samples=False,
    mean_field_slot='init_1',
    batch_size=200       # one batch = much faster
)
print("Posterior sampled!")
print(mod_spatial.samples['post_sample_means'].keys())

In [ ]:
# Extract spot factors (cell type abundances per cell)
spot_factors = mod_spatial.samples['post_sample_means']['spot_factors']
print(f"Spot factors shape: {spot_factors.shape}")  # (n_cells, n_cell_types)

# Add to adata_xenium
adata_xenium.obsm['cell_abundance'] = spot_factors

for i, ct in enumerate(inf_aver.columns):
    adata_xenium.obs[ct] = spot_factors[:, i]

print("Cell type abundances added to adata_xenium!")
print(adata_xenium.obs[inf_aver.columns[:5]].head())

In [ ]:
# Re-add spatial coordinates if lost during processing
if 'x_centroid' not in adata_xenium.obs.columns:
    cells = pd.read_csv(os.path.join(DATA_DIR, "cells.csv.gz"))
    cells = cells.set_index("cell_id")
    cells.index = cells.index.astype(str)
    adata_xenium.obs = adata_xenium.obs.join(cells[["x_centroid", "y_centroid"]], how="left")
    print("Spatial coords re-added.")
else:
    print("Spatial coords already present.")

## 6. Save Results

In [ ]:
# Save annotated AnnData — reload this in future sessions to skip retraining
out_path = os.path.join(RESULTS_DIR, "adata_xenium_cell2location.h5ad")
adata_xenium.write_h5ad(out_path)
print(f"Saved to: {out_path}")

# To reload in a future session:
# adata_xenium = sc.read_h5ad(out_path)

## 7. Visualize Cell Type Abundances Spatially

In [ ]:
# Plot spatial distribution of selected cell types
cell_types_to_plot = [
    'Astro_CTX', 'Oligo_2', 'Ext_L23', 'Ext_Hpc_CA1',
    'Micro', 'Inh_Sst', 'OPC_1', 'Ext_L56'
]

# Filter to only cell types that exist in our results
cell_types_to_plot = [ct for ct in cell_types_to_plot if ct in adata_xenium.obs.columns]

n_plots = len(cell_types_to_plot)
n_cols = 4
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

x = adata_xenium.obs['x_centroid']
y = adata_xenium.obs['y_centroid']

for i, ct in enumerate(cell_types_to_plot):
    abundance = adata_xenium.obs[ct]
    sc_plot = axes[i].scatter(
        x, y,
        c=abundance,
        cmap='viridis',
        s=0.5,
        vmin=0,
        vmax=np.percentile(abundance, 99)
    )
    axes[i].set_title(ct, fontsize=12)
    axes[i].axis('off')
    plt.colorbar(sc_plot, ax=axes[i], shrink=0.7, label='Abundance')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Cell Type Abundances — Xenium Mouse Brain (CTX + HP)', fontsize=16, y=1.01)
plt.tight_layout()

# Save BEFORE show
# First plot
plot_path = os.path.join(RESULTS_DIR, "cell_type_abundances_spatial.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"Plot saved to: {plot_path}")
plt.show()

# Second plot (all cell types)
all_ct_path = os.path.join(RESULTS_DIR, "all_cell_types_spatial.png")
plt.savefig(all_ct_path, dpi=120, bbox_inches='tight')
print(f"All cell types plot saved to: {all_ct_path}")
plt.show()

In [ ]:
# Bonus: Plot all 59 cell types in one figure
all_cts = inf_aver.columns.tolist()
n_cols = 6
n_rows = int(np.ceil(len(all_cts) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, ct in enumerate(all_cts):
    if ct not in adata_xenium.obs.columns:
        axes[i].set_visible(False)
        continue
    abundance = adata_xenium.obs[ct]
    axes[i].scatter(
        x, y,
        c=abundance,
        cmap='magma',
        s=0.3,
        vmin=0,
        vmax=np.percentile(abundance, 99)
    )
    axes[i].set_title(ct, fontsize=8)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('All Cell Types — Xenium Mouse Brain', fontsize=14, y=1.01)
plt.tight_layout()

all_ct_path = os.path.join(RESULTS_DIR, "all_cell_types_spatial.png")
plt.savefig(all_ct_path, dpi=120, bbox_inches='tight')
print(f"All cell types plot saved to: {all_ct_path}")
plt.show()

## Notes for Final Run

To get closer to publication-quality results, change the speed settings at the top of sections 4 and 5:

| Parameter | Testing | Final |
|-----------|---------|-------|
| `N_ITER_REG` | 100 | 4000 |
| `N_ITER_SPATIAL` | 500 | 20000 |
| `N_COMB` | 20 | 50 |
| `n_samples` (posterior) | 200 | 4000 |